# Byg billedgenereringsapplikationer (Azure OpenAI)

Der er mere i LLM'er end tekstgenerering. Du kan også generere billeder ud fra tekstbeskrivelser. Billeder som modalitet er nyttige inden for MedTech, arkitektur, turisme, spiludvikling, marketing og mere. I denne lektion arbejder vi med dagens **GPT Image** modeller på Microsoft Foundry.

## Læringsmål

Ved slutningen af denne lektion vil du kunne:

- Forklare, hvad billedgenerering er, og hvor det er nyttigt.
- Forstå `gpt-image` model-familien og hvordan den adskiller sig fra de tidligere DALL·E modeller.
- Bygge en billedgenereringsapplikation og redigere billeder.

## Hvad er billedgenerering?

Billedgenereringsmodeller skaber billeder ud fra en tekstprompt. Moderne modeller som `gpt-image` lærer sammenhængen mellem tekst og billeder under træning og omdanner derefter gentagne gange tilfældig støj til et billede, der matcher din prompt.

To velkendte familier af billedmodeller er:

- **`gpt-image` (OpenAI)** — den nuværende generation, der bruges i denne lektion. Den understøtter tekst-til-billede generering og billedredigering (inpainting med maske).
- **Midjourney** — en populær tredjepartsmodel med sin egen tjeneste og Discord-baseret workflow.

> De ældre OpenAI billedmodeller — **DALL·E 2** og **DALL·E 3** — er forældede. DALL·E 3 er ikke længere tilgængelig for nye udrulninger, og funktioner som `create_variation` eksisterede kun i DALL·E 2. Brug `gpt-image` modellerne til nye applikationer.

På Microsoft Foundry er **`gpt-image-2`** den nyeste og mest kapable billedmodel og anbefales som standard. `gpt-image-1.5` og `gpt-image-1-mini` er også generelt tilgængelige.

> **Vigtigt:** `gpt-image` modeller returnerer det genererede billede som **base64** (`b64_json`), ikke som en URL. Din kode afkoder base64-strengen til bytes og gemmer det — der er ingen billede-URL at downloade.


## Bygge din første billedgenereringsapplikation

Så hvad kræves der for at bygge en billedgenereringsapplikation? Du har brug for følgende biblioteker:

- **python-dotenv**, det anbefales stærkt at bruge dette bibliotek til at holde dine hemmeligheder i en *.env* fil adskilt fra koden.
- **openai**, dette bibliotek bruger du til at interagere med OpenAI API'en.
- **pillow**, til at arbejde med billeder i Python.

Hvis det ikke allerede er gjort, følg instruktionerne på [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) siden for at oprette en Microsoft Foundry ressourcer og model. Vælg **gpt-image-2** som modellen (den nyeste Azure OpenAI billedmodel; DALL·E er legacy).

1. Opret en fil *.env* med følgende indhold:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Find disse oplysninger i Microsoft Foundry portalen for din ressource under sektionen "Deployments".



1. Saml ovenstående biblioteker i en fil kaldet *requirements.txt* som følger:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Opret derefter et virtuelt miljø og installer bibliotekerne:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> For Windows, brug følgende kommandoer til at oprette og aktivere dit virtuelle miljø:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Tilføj følgende kode i en fil kaldet *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # konfigurer Azure OpenAI (Microsoft Foundry) klient.
    # Billedmodeller kræver en nyere API-version — tjek Foundry-dokumentationen for den, din model kræver.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Opret et billede ved at bruge billedgenererings-API'en
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Indtast din prompttekst her
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # f.eks. "gpt-image-2"
        )
        # Angiv mappen til det gemte billede
        image_dir = os.path.join(os.curdir, 'images')

        # Hvis mappen ikke findes, opret den
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialiser stien til billedet (bemærk at filtypen skal være png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image modeller returnerer billedet som base64 (b64_json), ikke en URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Vis billedet i standard billedfremviser
        image = Image.open(image_path)
        image.show()

    # fang undtagelser
    except BadRequestError as err:
        print(err)

    ```

Lad os forklare denne kode:

- Først importerer vi de biblioteker, vi har brug for, inklusive OpenAI-biblioteket, dotenv-biblioteket, base64-modulet og Pillow-biblioteket.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Dernæst indlæser vi miljøvariablerne fra *.env*-filen.

    ```python
    # import dotenv
    dotenv.load_dotenv()
    ```

- Derefter konfigurerer vi Azure OpenAI (Microsoft Foundry) klienten.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Dernæst genererer vi billedet:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Indtast din prompttekst her
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` modeller returnerer billedet som en **base64** streng i `data[0].b64_json`. Vi dekoder det til bytes og skriver det til en fil — der findes ingen URL til download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Til sidst åbner vi billedet og bruger standard billedfremviser til at vise det:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Flere detaljer om billedgenerering

Lad os se på parametrene for `images.generate`:

- **prompt**, er tekstprompten brugt til at generere billedet. Her er det "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, er størrelsen på det genererede billede (for eksempel `1024x1024`, `1536x1024`, `1024x1536`, eller `"auto"`).
- **n**, er antallet af genererede billeder. Her genererer vi ét.
- **model**, er navnet på din billedmodel-deployment (for eksempel `gpt-image-2`).

> Billedmodeller tager ikke en `temperature` parameter — det er en tekstgenereringskontrol. For at få variation, kald API'en igen; for at reducere variation, gør din prompt mere specifik.

## Yderligere funktioner ved billedgenerering

Du har set, hvordan man genererer et billede med få linjer Python. `gpt-image` modeller kan også **redigere** et eksisterende billede. Ved at levere et billede, en valgfri **maske** (som markerer området, der skal ændres), og en prompt, kan du ændre en del af et billede — for eksempel tilføje en hat på vores kanin.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# redigeringer returneres også som base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Basisbilledet indeholder kun kaninen; det endelige billede har hatten på kaninen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
